# Topic 17G — Hospital Records Cleaning Practice

This notebook gives extra practice with the hospital patient dataset used in the live-coding section.

**Main lesson link:** [Topic 17 — Data Cleaning with pandas](topic17_data_cleaning.ipynb)


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

filename = "data_cleaning_with_pandas/data/hospital_patients_messy.csv"
possible_paths = [Path(filename), Path("data") / filename, Path("../data") / filename, Path("../../data") / filename]
data_path = next(path for path in possible_paths if path.exists())
hosp = pd.read_csv(data_path)
print("Loaded:", data_path)
print(hosp.shape)
hosp.head()


---
## 1. Audit the Hospital Dataset


In [ ]:
hosp.info()
print()
print("Missing values:")
print(hosp.isna().sum())
print()
for col in ["region", "gender", "ward", "diagnosis"]:
    print("=" * 50)
    print(col)
    print(hosp[col].value_counts(dropna=False).head(12))


---
## 2. Clean Names, IDs, and Categories


In [ ]:
hosp["patient_name"] = hosp["patient_name"].str.strip()
hosp = hosp.drop_duplicates(subset="patient_id", keep="first")

hosp["region"] = hosp["region"].str.strip().str.title().replace({
    "Gr. Accra": "Greater Accra",
    "G.Accra": "Greater Accra",
})
hosp["ward"] = hosp["ward"].str.strip().str.title()
hosp["gender"] = hosp["gender"].str.strip().str.title().replace({"M": "Male", "F": "Female"})

print(hosp[["patient_id", "patient_name", "region", "gender", "ward"]].head())


---
## 3. Clean Age


In [ ]:
hosp["age"] = pd.to_numeric(hosp["age"], errors="coerce")
hosp.loc[(hosp["age"] < 0) | (hosp["age"] > 120), "age"] = np.nan
hosp["age"] = hosp["age"].fillna(hosp["age"].median()).round().astype(int)

print(hosp["age"].describe())


---
## 4. Parse Dates and Check Length of Stay


In [ ]:
hosp["admission_date"] = pd.to_datetime(hosp["admission_date"], dayfirst=True, errors="coerce")
hosp["discharge_date"] = pd.to_datetime(hosp["discharge_date"], dayfirst=True, errors="coerce")
hosp["length_of_stay"] = pd.to_numeric(hosp["length_of_stay"], errors="coerce")

expected_los = (hosp["discharge_date"] - hosp["admission_date"]).dt.days
hosp["los_mismatch"] = expected_los.notna() & hosp["length_of_stay"].notna() & (expected_los != hosp["length_of_stay"])

print("LOS mismatches:", hosp["los_mismatch"].sum())
print(hosp.loc[hosp["los_mismatch"], ["patient_id", "admission_date", "discharge_date", "length_of_stay"]].head())


---
## 5. Export a Clean Practice File


In [ ]:
output_path = "hospital_patients_clean_practice.csv"
hosp.to_csv(output_path, index=False)
print("Saved:", output_path)
print("Rows:", len(hosp))


### Useful Resources

- [pandas API: `to_datetime`](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html)
- [pandas user guide: Working with missing data](https://pandas.pydata.org/docs/user_guide/missing_data.html)
- [pandas API: `DataFrame.drop_duplicates`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html)


---
## Quick Reflection

1. Which hospital columns needed category cleaning?
2. Why should ages outside 0–120 be reviewed?
3. What does a length-of-stay mismatch suggest?
